# MA3001B • Semana 05: Procesos Estocásticos y Descomposición
## Estacionariedad Débil, Pruebas ADF/KPSS y Descomposición STL
**Profesor B: Modelos y TDA • Tecnológico de Monterrey**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alxcn/Models_MA3001B/blob/main/notebooks/N2_Estacionariedad_Descomposicion.ipynb)

---
### Objetivos de la Sesión
1. Distinguir formalmente entre proceso estocástico y realización muestral.
2. Aplicar e interpretar de forma cruzada las pruebas de hipótesis **ADF** y **KPSS**.
3. Estacionarizar series mediante operadores de diferencias ($\Delta, \Delta_s$) y transformaciones Box-Cox.
4. Descomponer series en tendencia, estacionalidad y residuo (aditiva, multiplicativa y **STL** robusto).
5. Reentrenar una red recurrente sobre la serie diferenciada e **invertir** la transformación para predecir el nivel original.


## 1. Problema Detonador: El Engaño del Split Aleatorio
Observemos cómo una regresión lineal evaluada con `train_test_split(shuffle=True)` reporta un $R^2$ artificialmente elevado ($>0.98$), pero falla por completo al extrapolar el futuro.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

y_det = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()

# Split con shuffle (MALO en series temporales)
X_idx = np.arange(len(y_det)).reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_idx, y_det.values, test_size=0.2, shuffle=True, random_state=42)
reg = LinearRegression().fit(X_tr, y_tr)

print(f"R^2 con shuffle aleatorio: {reg.score(X_te, y_te):.4f} (¡Aparenta ser casi perfecto!)")

# Pero al predecir el futuro real (corte cronológico):
corte = int(len(y_det) * 0.8)
reg_crono = LinearRegression().fit(X_idx[:corte], y_det.values[:corte])
pred_futuro = reg_crono.predict(X_idx[corte:])
rmse_futuro = np.sqrt(np.mean((y_det.values[corte:] - pred_futuro)**2))
print(f"RMSE real en el futuro: {rmse_futuro:.2f} ppm (La tendencia y los ciclos cambiaron)")


## 2. Marco Teórico Mínimo

### 2.1 Estacionariedad en Sentido Débil
$$1.\quad E[y_t] = \mu, \qquad 2.\quad \operatorname{Var}(y_t) = \sigma^2 < \infty, \qquad 3.\quad \operatorname{Cov}(y_t, y_{t+k}) = \gamma(k)$$

### 2.2 Matriz de Decisión Cruzada: ADF vs. KPSS
| ADF ($H_0$: raíz unitaria) | KPSS ($H_0$: estacionaria) | Diagnóstico | Acción |
| :---: | :---: | :--- | :--- |
| **Rechaza** ($p < 0.05$) | **No rechaza** ($p \ge 0.05$) | **Estacionaria** | Modelar directamente |
| **No rechaza** ($p \ge 0.05$) | **Rechaza** ($p < 0.05$) | **Raíz unitaria** | Diferenciar ($\Delta y_t$) |
| **Rechaza** ($p < 0.05$) | **Rechaza** ($p < 0.05$) | **Trend-stationarity** | Quitar tendencia |
| **No rechaza** ($p \ge 0.05$) | **No rechaza** ($p \ge 0.05$) | **Inconclusa** | Inspeccionar gráfica |

### 2.3 Descomposición de Series
$$\text{Aditiva: } y_t = T_t + S_t + R_t \qquad \text{vs.} \qquad \text{Multiplicativa: } y_t = T_t \times S_t \times R_t$$


### Configuración de Datos: Práctica (CO₂) o Serie del Reto
En la siguiente celda se carga la serie de práctica por defecto (**CO₂ mensual de Mauna Loa** de `statsmodels`).
Para aplicar el análisis a la serie de su equipo en el Reto, cambie la bandera `USAR_SERIE_RETO = True` y proporcione la ruta a su archivo CSV.

In [ ]:
# ==============================================================================
# CELDA SERIE_RETO: Alternar entre datos de práctica y datos del Reto
# ==============================================================================
USAR_SERIE_RETO = False  # Cambiar a True para usar la serie de su Reto

if USAR_SERIE_RETO:
    # Sustituya con la carga de la serie de su equipo:
    # df_reto = pd.read_csv("datos_reto.csv", index_col=0, parse_dates=True)
    # y = df_reto["columna_objetivo"].asfreq("MS").interpolate()
    print("Modo Reto activado.")
else:
    # Serie de práctica por defecto: CO2 de Mauna Loa (mensual interpolado)
    y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
    print(f"Cargada serie de práctica CO2: {len(y)} observaciones ({y.index.min().strftime('%Y-%m')} a {y.index.max().strftime('%Y-%m')})")


## 3. Ejemplo Aplicado Paso a Paso

### Paso 1: Diagnóstico de Estacionariedad Cruzado (ADF + KPSS)

In [ ]:
def reporte_estacionariedad(serie, nombre):
    s = serie.dropna()
    adf_res = adfuller(s, regression="c", autolag="AIC")
    kpss_res = kpss(s, regression="c", nlags="auto")
    
    adf_p = adf_res[1]
    kpss_p = kpss_res[1]
    
    # Diagnóstico automático
    if adf_p < 0.05 and kpss_p >= 0.05:
        diag = "Estacionaria"
    elif adf_p >= 0.05 and kpss_p < 0.05:
        diag = "No Estacionaria (Diferenciar)"
    elif adf_p < 0.05 and kpss_p < 0.05:
        diag = "Estacionaria alrededor de tendencia"
    else:
        diag = "Evidencia débil/inconclusa"
        
    print(f"{nombre:<28} | ADF p={adf_p:.4f} | KPSS p={kpss_p:.4f} | -> {diag}")

print("=== REPORTE DE ESTACIONARIEDAD ===")
reporte_estacionariedad(y, "1. Nivel original")
reporte_estacionariedad(y.diff(), "2. Primera diferencia (d=1)")
reporte_estacionariedad(y.diff().diff(12), "3. Dif. simple + Dif. estac. (d=1, D=1)")
reporte_estacionariedad(np.log(y).diff(), "4. Log + 1ª diferencia")


### Paso 2: Descomposición Clásica (Aditiva y Multiplicativa) vs. STL Robusto

In [ ]:
desc_add = seasonal_decompose(y, model="additive", period=12)
desc_mul = seasonal_decompose(y, model="multiplicative", period=12)
desc_stl = STL(y, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
axes[0].plot(y.index, y, label="Serie Original", color="#073642")
axes[0].set_title("Nivel Observado")
axes[1].plot(y.index, desc_stl.trend, label="Tendencia STL", color="#B58900")
axes[1].set_title("Componente de Tendencia (Loess)")
axes[2].plot(y.index, desc_stl.seasonal, label="Estacionalidad STL", color="#2AA198")
axes[2].set_title("Componente Estacional (Periodo=12)")
axes[3].scatter(y.index, desc_stl.resid, label="Residuo STL", color="#CB4B16", s=8, alpha=0.6)
axes[3].axhline(0, color="gray", linestyle="--")
axes[3].set_title("Residuo Irregular")

for ax in axes:
    ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Desviación estándar del residuo aditivo clásico: {desc_add.resid.std():.4f}")
print(f"Desviación estándar del residuo STL robusto:    {desc_stl.resid.std():.4f}")


### Paso 3: Reentrenar LSTM sobre Serie Estacionarizada e Invertir Transformación
Entrenamos la red sobre la primera diferencia $\Delta y_t = y_t - y_{t-1}$. Luego, reconstruimos el nivel mediante la suma acumulativa:
$$\hat{y}_t = y_{t-1} + \widehat{\Delta y}_t$$

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# 1) Serie estacionarizada (primera diferencia)
dy = y.diff().dropna()
n_tr = int(len(dy) * 0.8)

sc_diff = StandardScaler().fit(dy.iloc[:n_tr].to_frame())
z_diff = sc_diff.transform(dy.to_frame()).ravel().astype("float32")

L = 12
X_d, t_d = [], []
for i in range(len(z_diff) - L):
    X_d.append(z_diff[i:i + L])
    t_d.append(z_diff[i + L])
X_d = torch.tensor(np.array(X_d), dtype=torch.float32).unsqueeze(-1)
t_d = torch.tensor(np.array(t_d), dtype=torch.float32).unsqueeze(-1)

sp = n_tr - L
train_dl_diff = DataLoader(TensorDataset(X_d[:sp], t_d[:sp]), batch_size=32, shuffle=True)

class LSTMDiff(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

m_diff = LSTMDiff()
opt = torch.optim.Adam(m_diff.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for ep in range(35):
    m_diff.train()
    for xb, yb in train_dl_diff:
        opt.zero_grad()
        loss_fn(m_diff(xb), yb).backward()
        opt.step()

# Predicciones de la diferencia
m_diff.eval()
with torch.no_grad():
    pred_diff_norm = m_diff(X_d[sp:]).squeeze(-1).numpy()

pred_dy = sc_diff.inverse_transform(pred_diff_norm.reshape(-1, 1)).ravel()

# Inversión de la transformación: y[t] = y[t-1] + dy_hat[t]
# Índices del conjunto de prueba
test_indices = dy.index[n_tr:]
y_prev = y.loc[test_indices.shift(-1, freq="MS")].iloc[:len(pred_dy)].values  # y[t-1]
y_reconstruido = y.iloc[n_tr - 1 + np.arange(len(pred_dy))].values + pred_dy
y_real_test = y.iloc[n_tr + np.arange(len(pred_dy))].values

rmse_inv = np.sqrt(np.mean((y_real_test - y_reconstruido)**2))
print(f"RMSE en la escala original (ppm) con LSTM sobre serie estacionarizada: {rmse_inv:.4f}")


## 4. Actividad del Reto y Entregable

### Instrucciones:
1. Aplique las pruebas ADF y KPSS a su serie del Reto antes y después de transformar.
2. Evalúe si su serie requiere modelo aditivo o multiplicativo (revise si la oscilación crece con el nivel).
3. Complete las celdas `# TODO`.

In [ ]:
# ==============================================================================
# TODO 1: Diagnóstico ADF y KPSS sobre la serie del Reto
# ==============================================================================
# Imprima el reporte de estacionariedad para su serie y justifique qué orden d usará.


In [ ]:
# ==============================================================================
# TODO 2: Selección y justificación de descomposición (Aditiva vs Multiplicativa)
# ==============================================================================
# Compare la gráfica del residuo de ambas descomposiciones y justifique su elección.


## 5. Conclusiones y Pregunta Abierta
- **Conclusión:** Diferenciar estabiliza la media y permite a los modelos de machine learning aprender oscilaciones sin verse superados por tendencias no acotadas.
- **Pregunta para Semana 06:** Ahora que la serie es estacionaria, ¿cuántos rezagos tienen correlación real (parcial) con el presente y cuál es el periodo exacto del ciclo oscilatorio?
